# 11 - Foundry IQ knowledge base: existing indexes + Web IQ

## Required permissions

| Identity | Permission | Function |
|---|---|---|
| Signed-in user | **Search Service Contributor** on Search | Create knowledge sources and the knowledge base |
| Signed-in user | **Search Index Data Reader** (or **Search Index Data Contributor**) on Search, plus access to the indexed SharePoint documents | Retrieve indexed content with SharePoint permissions enforced |
| Source author | Web IQ profile API key | Store the key in this knowledge source for notebook and portal retrieval |
| Search managed identity | **Cognitive Services User on the parent Foundry resource** | Call the chat model and existing index vectorizers |

**Resource names:** keep the same `WORKSHOP_PREFIX` used in earlier notebooks. This notebook creates `{prefix}-kb-blob`, `{prefix}-kb-sharepoint`, `{prefix}-kb-webiq` and `{prefix}-kb`. Notebooks 12 and 14 reuse these names. Prefixes prevent naming collisions, not enforce access isolation.

Build **one Foundry IQ knowledge base** over the content already indexed in earlier notebooks, plus live Web IQ:

| Knowledge source | Kind | Existing content |
|---|---|---|
| Blob documents | `searchIndex` | Notebook 05's Content Understanding index, `{prefix}-cu-chunks` |
| SharePoint documents | `searchIndex` | Notebook 07's `{prefix}-nb7-index`, including its Entra user/group ACLs |
| Web IQ | `mcpServer` | Live public web through `https://api.microsoft.ai/v3/mcp` |

The two `searchIndex` knowledge sources are **references to existing indexes**. This notebook does not upload documents, create indexes, change index schemas, or create/run data sources, skillsets or indexers. It does not use remote SharePoint or depend on notebook 08.

Set the kernel working directory to `c-sharp-notebooks`. Configuration loads `..\.env` first, then `..\.env.azure`, both with `override=True`; set `WORKSHOP_PREFIX` before running (no prefix prompt).


## Prerequisites

- Complete notebooks **05** (Content Understanding) and **07** using the same Search service and `WORKSHOP_PREFIX`. Their indexes must already contain content you can retrieve. Indexing and SharePoint ingestion-app setup stay in those notebooks.
- Keep the deployment settings in `.env.azure` and `WORKSHOP_PREFIX` plus `WEBIQ_API_KEY` in `.env`. Sign in with `az login` in the deployment tenant.
- The Search managed identity must be able to call the deployed chat model and the existing indexes' embedding vectorizers. Ensure it has **Cognitive Services OpenAI User** or the broader **Cognitive Services User on the parent Foundry resource**.
- No Blob Storage access, SharePoint ingestion-app credentials, or Microsoft 365 Copilot license is needed to run this notebook. SharePoint document access is still required; an existing index does not bypass its ACLs.

These are preview APIs (`2026-08-01-preview`). Retrieval and Web IQ requests can incur charges, but this notebook starts no indexing work.

Progression: [09 - Web-only](09-foundry-iq-webiq-only.ipynb) -> [10 - Web + Blob](10-foundry-iq-webiq-blob.ipynb) -> this **Web + Blob + SharePoint** lab. This lab retains its existing `<prefix>-kb-*` sources and `<prefix>-kb` knowledge base, used by notebooks 12 and 14; it does not modify the independent 09/10 definitions.


## C# configuration and inline REST helpers

Use the C# kernel in Polyglot Notebooks / .NET Interactive, with the working directory
set to `c-sharp-notebooks`. This notebook is self-contained: run this setup cell,
then any numbered step independently once its Azure prerequisites exist. The helpers
below use Azure CLI authentication, standard .NET HTTP/JSON and no NuGet packages.
Setup does not acquire tokens or call Azure. The numbered cells show actual REST
payloads, not calls into the console app.

The `.env` then `.env.azure` files in the parent directory override process settings.
No prefix prompt or infrastructure detection is performed. Errors redact the loaded
Web IQ key and the current Search token before logging. Never display settings or
the Web IQ source payload: it contains the stored key.


In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Threading;
using System.Threading.Tasks;
using System.Net.Http;
using System.Diagnostics;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;

public sealed class KnowledgeBaseSettings(string tenant, string searchEndpoint, string account,
    string deployment, string model, string prefix, string webIqKey)
{
    public string Tenant { get; } = tenant;
    public string SearchEndpoint { get; } = searchEndpoint;
    public string Account { get; } = account;
    public string Deployment { get; } = deployment;
    public string Model { get; } = model;
    public string Prefix { get; } = prefix;
    public string WebIqKey { get; } = webIqKey;
    public string ModelEndpoint => $"https://{Account}.openai.azure.com";
    public string BlobIndexName => Prefix + "-cu-chunks";
    public string SharePointIndexName => Prefix + "-nb7-index";
    public string BlobSourceName => Prefix + "-kb-blob";
    public string SharePointSourceName => Prefix + "-kb-sharepoint";
    public string WebIqSourceName => Prefix + "-kb-webiq";
    public string BaseName => Prefix + "-kb";

    public static KnowledgeBaseSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name + ". See .env.example.");
        var settings = new KnowledgeBaseSettings(Required("AZURE_TENANT_ID"), Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'),
            Required("AZURE_AI_ACCOUNT_NAME"), Required("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
            Environment.GetEnvironmentVariable("AZURE_AI_MODEL_NAME") ?? "gpt-5-mini",
            Required("WORKSHOP_PREFIX"), Required("WEBIQ_API_KEY"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and 3-20 lowercase letters/digits prefix starting with a letter.");
        if (!Uri.TryCreate(SearchEndpoint, UriKind.Absolute, out var endpoint) || endpoint.Scheme != "https" ||
            endpoint.UserInfo != "" || !endpoint.IsDefaultPort || endpoint.AbsolutePath != "/" ||
            endpoint.Query != "" || endpoint.Fragment != "" ||
            !Regex.IsMatch(endpoint.Host, @"^[a-z0-9][a-z0-9-]*\.search\.windows\.net$", RegexOptions.IgnoreCase))
            throw new InvalidOperationException("Use a public Azure HTTPS Search service root without credentials, path or query.");
        if (!Regex.IsMatch(Account, @"^[a-zA-Z0-9][a-zA-Z0-9-]*$") ||
            string.IsNullOrWhiteSpace(Deployment) || string.IsNullOrWhiteSpace(Model) || string.IsNullOrWhiteSpace(WebIqKey))
            throw new InvalidOperationException("Use a Foundry account name, chat deployment/model and non-empty WEBIQ_API_KEY.");
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith('#')) continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal)) setting = setting[7..].TrimStart();
            int equals = setting.IndexOf('=');
            if (equals <= 0) continue;
            string name = setting[..equals].Trim();
            string value = setting[(equals + 1)..].Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[^1] == '"') || (value[0] == '\'' && value[^1] == '\'')))
                value = value[1..^1];
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0) value = value[..comment].TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

public sealed class KnowledgeBaseTransport(HttpClient client, string tenant, Func<Task<string>>? tokenProvider = null)
{
    public const string Scope = "https://search.azure.com/.default";
    public const string ApiVersion = "2026-08-01-preview";
    readonly Func<Task<string>> getToken = tokenProvider ?? (() => AzureCliTokenAsync(tenant));
    public JsonObject? LastResult { get; private set; }
    public int LastStatus { get; private set; }

    public async Task<JsonObject> SendAsync(KnowledgeBaseSettings settings, HttpMethod method, string path,
        JsonObject? body = null, bool allowMissing = false, bool userContext = false,
        bool diagnostics = false, int timeoutSeconds = 90)
    {
        settings.Validate();
        bool retrieve = path.EndsWith("/retrieve", StringComparison.Ordinal);
        if (retrieve) { LastResult = null; LastStatus = 0; }
        string token;
        try { token = await getToken(); }
        catch (Exception error)
        {
            throw new InvalidOperationException(KnowledgeBaseResults.Redact(error.Message, settings.WebIqKey));
        }
        using var request = new HttpRequestMessage(method, $"{settings.SearchEndpoint}/{path}?api-version={ApiVersion}");
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", token);
        if (userContext) request.Headers.Add("x-ms-query-source-authorization", token);
        if (body is not null)
            request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(timeoutSeconds));
        string Safe(string value) => KnowledgeBaseResults.Redact(value, settings.WebIqKey, token);
        try
        {
            using var response = await client.SendAsync(request, timeout.Token);
            string text = await response.Content.ReadAsStringAsync(timeout.Token);
            int status = (int)response.StatusCode;
            Console.WriteLine($"HTTP status: {status}");
            Console.WriteLine("Request ID: " + Safe(response.Headers.TryGetValues("x-ms-request-id", out var ids)
                ? string.Join(", ", ids) : response.Headers.TryGetValues("request-id", out ids) ? string.Join(", ", ids) : ""));
            if (allowMissing && status == 404) return new JsonObject();
            if (!response.IsSuccessStatusCode)
            {
                Console.Error.WriteLine($"HTTP {status}: Search request failed. UTC (Date): {response.Headers.Date:R}");
                Console.Error.WriteLine(status switch
                {
                    401 => "Check Azure CLI sign-in and tenant; this is Search authorization, not proof of Web IQ key failure.",
                    403 => "Check Search RBAC/network access and document ACLs; model/vectorizer errors need Search managed-identity inference access.",
                    429 => "Quota or throttling; inspect Retry-After and capacity/profile usage. No automatic billable retry.",
                    404 when path.StartsWith("indexes/") => "Complete notebooks 05 and 07 with the same prefix; do not create an index here.",
                    _ => "Inspect activity/request IDs; auth_missing_token concerns the stored Web IQ profile header."
                });
                foreach (var header in response.Headers)
                    if (header.Key.Contains("request", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Contains("correlation", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Contains("trace", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Equals("retry-after", StringComparison.OrdinalIgnoreCase))
                        Console.Error.WriteLine(header.Key + ": " + Safe(string.Join(", ", header.Value)));
                string safe = Safe(text);
                Console.Error.WriteLine("Response body: " + safe[..Math.Min(8000, safe.Length)]);
                throw new HttpRequestException($"HTTP {status}: Search request failed; inspect redacted diagnostics above.", null, response.StatusCode);
            }
            JsonObject result;
            try
            {
                result = string.IsNullOrWhiteSpace(text) && method != HttpMethod.Post ? new JsonObject()
                    : JsonNode.Parse(Safe(text)) as JsonObject ?? throw new JsonException();
            }
            catch (JsonException)
            {
                throw new InvalidOperationException("Search returned invalid JSON; no complete result available.");
            }
            if (retrieve)
            {
                LastResult = result;
                LastStatus = status;
                KnowledgeBaseResults.Print(result, status, diagnostics, settings.WebIqKey, token);
            }
            else if (status == 206)
                throw new InvalidOperationException("Search returned a partial response; no complete result available.");
            return result;
        }
        catch (HttpRequestException error)
        {
            throw new HttpRequestException(Safe(error.Message), null, error.StatusCode);
        }
        catch (OperationCanceledException)
        {
            throw new TimeoutException("Search request timed out before a complete response; inspect network/service health. No automatic retry.");
        }
    }

    public static async Task<string> AzureCliTokenAsync(string tenant)
    {
        if (!Guid.TryParseExact(tenant, "D", out _)) throw new InvalidOperationException("Invalid tenant GUID.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true
        };
        if (OperatingSystem.IsWindows())
            foreach (string argument in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(argument);
        foreach (string argument in new[] { "account", "get-access-token", "--scope", Scope, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(argument);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out before Search/Web IQ. Run az login --tenant AZURE_TENANT_ID.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed before Search/Web IQ. Run az login --tenant AZURE_TENANT_ID.");
        try
        {
            return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
                ? token : throw new JsonException();
        }
        catch (Exception error) when (error is JsonException or InvalidOperationException)
        {
            throw new InvalidOperationException("Azure CLI returned no usable access token.");
        }
    }
}

public static class KnowledgeBaseResults
{
    public static string Redact(string text, params string[] secrets)
    {
        foreach (string secret in secrets.Where(secret => !string.IsNullOrEmpty(secret)))
        {
            text = text.Replace(secret, "<redacted>", StringComparison.Ordinal);
            string escaped = secret;
            for (int level = 0; level < 3; level++)
            {
                escaped = JsonSerializer.Serialize(escaped)[1..^1];
                text = text.Replace(escaped, "<redacted>", StringComparison.Ordinal);
            }
            text = text.Replace(Uri.EscapeDataString(secret), "<redacted>", StringComparison.Ordinal);
        }
        return text;
    }

    public static void ValidateQuery(string question, string outputMode)
    {
        if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
        if (outputMode is not ("answerSynthesis" or "extractiveData"))
            throw new ArgumentException("Use answerSynthesis or extractiveData.");
    }

    public static JsonObject ObjectData(JsonNode? node)
    {
        if (node is JsonObject obj) return (JsonObject)obj.DeepClone();
        if (node is JsonValue value && value.TryGetValue<string>(out string? text))
        {
            try { return JsonNode.Parse(text) as JsonObject ?? new JsonObject(); }
            catch (JsonException) { }
        }
        return new JsonObject();
    }

    public static string FormatReference(JsonObject reference)
    {
        var data = ObjectData(reference["sourceData"]);
        foreach (var property in ObjectData(data["content"]))
            data[property.Key] = property.Value?.DeepClone();
        var source = ObjectData(data["source"]);
        string First(params JsonNode?[] values) => values.Select(v => v?.ToString()).FirstOrDefault(v => !string.IsNullOrEmpty(v)) ?? "";
        string title = First(data["source/file_name"], source["file_name"], data["sourceFile"],
            data["title"], reference["docKey"]);
        if (title.Length == 0) title = "Source";
        string url = First(data["source/url"], source["url"], data["webUrl"], data["url"]);
        JsonNode? firstNode = data.ContainsKey("page_from") ? data["page_from"] : data["page"];
        JsonNode? lastNode = data.ContainsKey("page_to") ? data["page_to"] : data["pageTo"];
        string pages = "";
        if (firstNode is JsonValue firstValue && firstValue.TryGetValue<int>(out int first) &&
            lastNode is JsonValue lastValue && lastValue.TryGetValue<int>(out int last) && first >= 1 && first <= last)
        {
            pages = first == last ? $"p. {first}" : $"pp. {first}-{last}";
            if (url.Length > 0) url = url.Split('#')[0] + $"#page={first}";
        }
        return $"- [ref_id:{reference["id"]}] {title} {pages} {(url.Length > 0 ? url : "(No source URL returned.)")}";
    }

    public static void Diagnostics(JsonObject? result, int status, params string[] secrets)
    {
        if (result is null) { Console.WriteLine("Run a query first; no retrieval response exists in this process."); return; }
        void Write(JsonNode node) => Console.WriteLine(Redact(node.ToJsonString(new JsonSerializerOptions { WriteIndented = true }), secrets));
        Console.WriteLine($"HTTP status: {status}");
        Console.WriteLine("Full activity:");
        Write(result["activity"] ?? new JsonArray());
        Console.WriteLine("Web IQ activity only:");
        Write(new JsonArray((result["activity"] as JsonArray ?? new JsonArray())
            .OfType<JsonObject>().Where(s => s["type"]?.ToString() == "mcpServer").Select(s => s.DeepClone()).ToArray()));
        Console.WriteLine("Raw references:");
        Write(result["references"] ?? new JsonArray());
    }

    public static void Print(JsonObject result, int status, bool diagnostics = false, params string[] secrets)
    {
        void Write(string value) => Console.WriteLine(Redact(value, secrets));
        if (diagnostics) Diagnostics(result, status, secrets);
        if (result["activity"] is not JsonArray activity ||
            activity.Any(s => s is not JsonObject) ||
            result["response"] is not JsonArray messages || messages.Count == 0 ||
            messages.Any(m => m is not JsonObject || m["content"] is not JsonArray content ||
                content.Count == 0 || content.Any(p => p is not JsonObject || p["text"] is not JsonValue text ||
                    !text.TryGetValue<string>(out _))) ||
            result["references"] is not null && (result["references"] is not JsonArray refs || refs.Any(r => r is not JsonObject)))
            throw new InvalidOperationException("Malformed retrieval response; no complete result available.");
        Console.WriteLine("\nAnswer or extractive output:");
        foreach (JsonObject message in messages.OfType<JsonObject>())
            foreach (JsonObject part in message["content"]!.AsArray().OfType<JsonObject>())
                Write(part["text"]!.ToString());
        Console.WriteLine("\nReferences:");
        foreach (JsonObject reference in (result["references"] as JsonArray ?? new JsonArray()).OfType<JsonObject>())
            Write(FormatReference(reference));
        Console.WriteLine("\nActivity:");
        foreach (JsonObject step in activity.OfType<JsonObject>())
            Write($"- {step["type"]} {step["knowledgeSourceName"]} {step["count"]} {step["error"]}");
        if (status == 206 || activity.OfType<JsonObject>().Any(s =>
            s["error"] is JsonNode error && error.ToJsonString() is not ("null" or "{}" or "[]" or "\"\"" or "false" or "0")))
            throw new InvalidOperationException("Retrieval was partial or a source failed. Inspect activity; use --diagnostics for raw existing response details.");
        if (result["references"] is not JsonArray references || references.Count == 0)
            Console.WriteLine("No references returned. Inspect activity; this alone does not establish an authentication failure.");
    }
}


KnowledgeBaseSettings settings = KnowledgeBaseSettings.Load(Path.GetFullPath(".."));
HttpClient client = new HttpClient(new HttpClientHandler { AllowAutoRedirect = false })
{
    Timeout = TimeSpan.FromSeconds(150)
};
Func<Task<string>> tokenProvider = () => KnowledgeBaseTransport.AzureCliTokenAsync(settings.Tenant);
KnowledgeBaseTransport transport = new KnowledgeBaseTransport(client, settings.Tenant, () => tokenProvider());
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("Existing indexes: " + settings.BlobIndexName + " + " + settings.SharePointIndexName);
Console.WriteLine("Knowledge base: " + settings.BaseName);


## 1. Check the existing indexes

The next cell reads both schemas and checks how many chunks are visible to your signed-in user. Missing indexes or zero visible chunks must be resolved in notebooks 05 or 07, not by creating another ingestion pipeline here.

The Blob indexes from 04 and 05 share the same content and source fields, so change only the `BlobIndexName` property in Configuration to select between them. The SharePoint source expects notebook 07's Content Understanding index with `UserIds` and `GroupIds` permission fields.

In [ ]:
foreach (var (indexName, lab, semanticName, contentField, required) in new[]
{
    (settings.BlobIndexName, "05", "chunk-semantic", "chunk",
        new[] { "chunk_id", "chunk", "title", "source/file_name", "source/url", "page_from", "page_to" }),
    (settings.SharePointIndexName, "07", "semantic-config", "content",
        new[] { "id", "content", "sourceFile", "webUrl", "page", "pageTo" })
})
{
    JsonObject index = await transport.SendAsync(settings, HttpMethod.Get, $"indexes/{indexName}");
    var fields = new Dictionary<string, JsonNode>();
    foreach (JsonObject field in (index["fields"] as JsonArray ?? new JsonArray()).OfType<JsonObject>())
    {
        if (field["name"] is not JsonValue name) continue;
        fields[name.ToString()] = field;
        foreach (JsonObject child in (field["fields"] as JsonArray ?? new JsonArray()).OfType<JsonObject>())
            if (child["name"] is JsonValue childName) fields[$"{name}/{childName}"] = child;
    }
    if (required.Any(name => !fields.TryGetValue(name, out var field) || field["retrievable"]?.ToString() == "false") ||
        fields[contentField]["searchable"]?.ToString() != "true")
        throw new InvalidOperationException($"{indexName} does not have the expected retrievable content/source fields. See notebook {lab}.");
    if (!(index["semantic"]?["configurations"] as JsonArray ?? new JsonArray())
        .Any(config => config?["name"]?.ToString() == semanticName))
        throw new InvalidOperationException($"{indexName} is missing semantic configuration {semanticName}. See notebook {lab}.");
    bool sharePoint = indexName == settings.SharePointIndexName;
    if (sharePoint && (index["permissionFilterOption"]?.ToString() != "enabled" ||
        fields.GetValueOrDefault("UserIds")?["permissionFilter"]?.ToString() != "userIds" ||
        fields.GetValueOrDefault("GroupIds")?["permissionFilter"]?.ToString() != "groupIds"))
        throw new InvalidOperationException("The SharePoint index must retain notebook 07 user/group permission filtering.");
    JsonObject countResult = await transport.SendAsync(settings, HttpMethod.Post, $"indexes/{indexName}/docs/search",
        new JsonObject { ["search"] = "*", ["count"] = true, ["top"] = 0 }, userContext: sharePoint);
    if (countResult["@odata.count"] is not JsonValue countValue ||
        !countValue.TryGetValue<long>(out long count) || count <= 0)
        throw new InvalidOperationException($"No chunks are visible in {indexName}. Check ingestion and document access in notebook {lab}.");
    Console.WriteLine($"{indexName}: {count} visible chunks; existing schema unchanged.");
}

## 2. Reference the Blob index

`searchIndexParameters.searchIndexName` points at the index that already exists. `searchFields` selects its text for retrieval; `sourceDataFields` returns chunk text and source metadata with references. Search uses the existing index's semantic configuration and vectorizer. No storage connection or new embedding deployment is configured here.

In [ ]:
JsonObject blobSource = new()
    {
        ["name"] = settings.BlobSourceName,
        ["kind"] = "searchIndex",
        ["description"] = "Travel reimbursement and onboarding policy documents from the existing Blob index.",
        ["searchIndexParameters"] = new JsonObject
        {
            ["searchIndexName"] = settings.BlobIndexName,
            ["semanticConfigurationName"] = "chunk-semantic",
            ["searchFields"] = new JsonArray(new JsonObject { ["name"] = "chunk" }),
            ["sourceDataFields"] = new JsonArray(
                new JsonObject { ["name"] = "chunk_id" }, new JsonObject { ["name"] = "chunk" },
                new JsonObject { ["name"] = "title" }, new JsonObject { ["name"] = "source/file_name" },
                new JsonObject { ["name"] = "source/url" }, new JsonObject { ["name"] = "page_from" },
                new JsonObject { ["name"] = "page_to" })
        }
};
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.BlobSourceName}", blobSource);
Console.WriteLine("Ready: " + settings.BlobSourceName);

## 3. Reference the SharePoint index

This is also a `searchIndex` knowledge source, **not** `indexedSharePoint` (which would build another ingestion pipeline) or `remoteSharePoint` (which would call Microsoft 365 at query time).

The existing index retains its `permissionFilterOption`, `UserIds` and `GroupIds`. Retrieval sends the signed-in user's Search-scoped token in `x-ms-query-source-authorization`; Search applies the existing ACLs. ACL fields are not returned in source data. Source links and page ranges are returned for citations.

In [ ]:
JsonObject sharePointSource = new()
    {
        ["name"] = settings.SharePointSourceName,
        ["kind"] = "searchIndex",
        ["description"] = "Reports and documents from the existing SharePoint library index, with Entra user/group permissions.",
        ["searchIndexParameters"] = new JsonObject
        {
            ["searchIndexName"] = settings.SharePointIndexName,
            ["semanticConfigurationName"] = "semantic-config",
            ["searchFields"] = new JsonArray(new JsonObject { ["name"] = "content" }),
            ["sourceDataFields"] = new JsonArray(
                new JsonObject { ["name"] = "id" }, new JsonObject { ["name"] = "content" },
                new JsonObject { ["name"] = "sourceFile" }, new JsonObject { ["name"] = "webUrl" },
                new JsonObject { ["name"] = "page" }, new JsonObject { ["name"] = "pageTo" })
        }
};
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.SharePointSourceName}", sharePointSource);
Console.WriteLine("Ready: " + settings.SharePointSourceName);

## 4. Web IQ knowledge source (MCP server)

Web IQ is a remote knowledge source: Search calls its MCP tools at query time and ranks the results next to your private content. The model in the knowledge base chooses which tool to call and writes its arguments.

**The workshop Web IQ key is stored in this knowledge source** using `storedHeaders` authentication. Search sends `x-apikey` during MCP tool discovery and calls, so the Foundry IQ portal works without supplying custom headers on each question. Search masks stored header values when the source is read back.

Use a workshop-only key: authorized knowledge-base callers share its quota and billing. Restrict source-authoring permissions, rotate/revoke the key after use, and never print the `webiq_source` variable or share outputs containing credentials. The key is loaded from `.env`, not embedded in notebook source. Notebook 12 separately demonstrates keeping a key in APIM instead.

Only the basic **`web` search tool** is configured; the Web IQ Auto tool is not used. `outputParsing: {'kind': 'auto'}` below is Search's MCP response-parsing mode, not a Web IQ tool.

The source description and retrieval instructions tell the planner to use **`contentFormat: passage`**: query-dependent search passages, not full-document text, HTML or Markdown. This works with passage-only profiles; requesting full-document formats can return 403. The tool arguments are visible under `transport.LastResult["activity"]` if a call fails.

Preview limits for MCP knowledge sources: `minimal` reasoning effort and `alwaysQuerySource` are not supported, and tool calls can be slow, so the retrieve request sets `maxRuntimeInSeconds`.

In [ ]:
JsonObject webIqSource = new()
    {
        ["name"] = settings.WebIqSourceName,
        ["kind"] = "mcpServer",
        ["description"] = "Basic Web IQ web search for public information. Always set contentFormat to passage; do not request full-document text, html or markdown.",
        ["resultsProcessing"] = "rerank",
        ["mcpServerParameters"] = new JsonObject
        {
            ["serverURL"] = "https://api.microsoft.ai/v3/mcp",
            ["authentication"] = new JsonObject
            {
                ["kind"] = "storedHeaders",
                ["storedHeadersParameters"] = new JsonObject
                {
                    ["headers"] = new JsonObject { ["x-apikey"] = settings.WebIqKey }
                }
            },
            ["tools"] = new JsonArray(new JsonObject
            {
                ["name"] = "web",
                ["outputParsing"] = new JsonObject { ["kind"] = "auto" },
                ["maxOutputTokens"] = 2000
            })
        }
};
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.WebIqSourceName}", webIqSource, timeoutSeconds: 120);
Console.WriteLine("Ready: " + settings.WebIqSourceName);

## 5. Create the knowledge base

The knowledge base lists the three sources and the model that plans retrieval. For each question the model decides which sources are worth querying, writes a subquery per source, and Search merges and reranks everything into one result. With `answerSynthesis`, the same model then writes a cited answer from the retrieved evidence.

- `retrievalInstructions` tells the planner what each source is for. Clear instructions matter more than anything else here.
- `retrievalReasoningEffort` is `medium`: the Web IQ (MCP) source does not support `minimal`.
- The model is a Foundry deployment that the Search managed identity calls (Cognitive Services User); no model keys are used.

In [ ]:
JsonObject knowledgeBase = new()
    {
        ["name"] = settings.BaseName,
        ["description"] = "Workshop policies (Blob and SharePoint) plus live public web information from Web IQ.",
        ["retrievalInstructions"] = $"Use {settings.BlobSourceName} for travel reimbursement and onboarding policy documents from Blob. " +
            $"Use {settings.SharePointSourceName} for reports and other documents from the SharePoint library. " +
            "Consider both indexed sources when an internal question does not identify its document source. " +
            $"Use {settings.WebIqSourceName} for current public information and external facts. " +
            "For every Web IQ web call, explicitly set contentFormat to passage. " +
            "Never request text, html or markdown content formats; use only query-dependent search passages. " +
            "For internal-versus-public comparisons, query the relevant internal source and Web IQ. " +
            "Do not send internal document text to Web IQ.",
        ["answerInstructions"] = "Answer concisely and cite every source. Say which facts come from internal documents and which " +
            "come from the web. If the sources do not contain the answer, say you do not know.",
        ["outputMode"] = "answerSynthesis",
        ["knowledgeSources"] = new JsonArray(new JsonObject { ["name"] = settings.BlobSourceName },
            new JsonObject { ["name"] = settings.SharePointSourceName }, new JsonObject { ["name"] = settings.WebIqSourceName }),
        ["models"] = new JsonArray(new JsonObject
        {
            ["kind"] = "azureOpenAI",
            ["azureOpenAIParameters"] = new JsonObject
            {
                ["resourceUri"] = settings.ModelEndpoint,
                ["deploymentId"] = settings.Deployment,
                ["modelName"] = settings.Model
            }
        }),
        ["retrievalReasoningEffort"] = new JsonObject { ["kind"] = "medium" }
};
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgebases/{settings.BaseName}", knowledgeBase);
Console.WriteLine("Ready: " + settings.BaseName);

### Use the knowledge base from the Foundry IQ portal

1. Set the workshop `WEBIQ_API_KEY` in your local `.env`; do not paste it into the chat or notebook source.
2. Run **Configuration**, then **4. Web IQ knowledge source** (`create-webiq-source`). Its PUT saves `mcpServerParameters.authentication.kind = storedHeaders` with the `x-apikey` header on `{prefix}-kb-webiq` in Azure AI Search. On a first run, also complete section 5 to create the knowledge base.
3. Open the existing `{prefix}-kb` knowledge base in the Foundry IQ portal. Refresh it and start a new chat after updating authentication. Updating the source does not require recreating the knowledge base or either index.
4. Ask a public-web question such as **What is the latest stable release of Python, and when was it released?** Check that Web IQ activity succeeds and the answer includes references, then try the internal/web comparison below.

**Why this is needed:** the portal does not supply the notebook's former per-request Web IQ control headers. Without source authentication, MCP tool discovery returns **401 `auth_missing_token`** even when notebook requests that supply a key succeed. A model saying it cannot browse is not the diagnosis; check the retrieval activity/error.

The credential lives on the **Search knowledge source**, not in the portal chat. Search masks it on readback and supplies it to Web IQ automatically. Source callers share its quota/billing, but each user's SharePoint document access is still enforced separately. To rotate the key, update `.env` and rerun Configuration plus `create-webiq-source`.

See [MCP knowledge-source authentication options](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server#authentication-options).

## 6. Ask questions

Each request carries your Search identity:

- `Authorization`: your token for Azure AI Search.
- `x-ms-query-source-authorization`: your token again, so SharePoint permissions are evaluated as **you**, not as the service.

Search supplies the stored Web IQ key from the knowledge source. Retrieve requests need no Web IQ control headers; the same source can therefore be queried from the Foundry IQ portal. Refresh the portal after creating or updating the source, then start a new chat to avoid retaining an earlier failed answer.

`includeActivity` returns what each source did, so you can see which sources the planner actually used. The questions name sources to make this first check explicit; source selection is still the model's judgment, not a guarantee. After verifying each source, try questions without source names.

Change these example questions to match your indexed documents. A source may be skipped by the planner; inspect activity and references rather than assuming all three sources were used.

The reference formatter below displays the document filename, Content Understanding page range and source URL instead of only a chunk key. `page_from` and `page_to` are returned by the Blob source's `sourceDataFields`; rerun `create-blob-source` after adding them. A range such as 1-2 means the chunk spans both pages, not that every claim appears on both. The PDF link uses `#page=` to open the first page in viewers that support it. Private Blob URLs still require authenticated access; these links do not grant access or create a SAS token. Missing metadata falls back to the reference/chunk identifier without inventing a page.

Web IQ also needs `includeReferenceSourceData: True` to return evidence URLs, not just `[ref_id:N]` markers. Its title and URL can be encoded as JSON inside `sourceData.content`; the formatter reads those fields. If no URL is returned, it says so rather than inventing a link. Rerun the formatter and query cells after this change; no knowledge-source update is needed for web links.

### 6a. An internal question

### Readable references and output mode

The inline `KnowledgeBaseResults.FormatReference` helper in Configuration handles every
source in the same way, including mixed results, encoded `sourceData` and encoded Web IQ
`content`. It shows `[ref_id:N]`, the returned filename/title, valid page ranges and source
URL. It does not invent URLs/pages. The query cells below all request reference source
data from **all three sources**; naming a source in the question does not force routing.

Leave `answerSynthesis` for the stored default, or select `extractiveData` per request.
Extractive output still uses model planning and can incur charges. This does not update
the stored KB. Setup and this cell make no retrieval call.


In [ ]:
string outputMode = "answerSynthesis"; // Or "extractiveData".
KnowledgeBaseResults.ValidateQuery("mode check", outputMode);

In [ ]:
{
    string question = $"Using {settings.BlobSourceName}, what receipts do I need to submit for travel reimbursement?";
    KnowledgeBaseResults.ValidateQuery(question, outputMode);
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 60,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName,
                ["kind"] = "mcpServer",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            })
    };
    // Python uses the stored answerSynthesis default. Override only when requested.
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    await transport.SendAsync(settings, HttpMethod.Post, $"knowledgebases/{settings.BaseName}/retrieve",
        payload, userContext: true, timeoutSeconds: 120);
}

### 6b. A question only the web can answer

Internal documents cannot answer this. A good run shows the Web IQ source used and the internal sources skipped or returning nothing relevant.

In [ ]:
{
    string question = "What is the weather in St. Paul MN today?";
    KnowledgeBaseResults.ValidateQuery(question, outputMode);
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 60,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName,
                ["kind"] = "mcpServer",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            })
    };
    // Python uses the stored answerSynthesis default. Override only when requested.
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    await transport.SendAsync(settings, HttpMethod.Post, $"knowledgebases/{settings.BaseName}/retrieve",
        payload, userContext: true, timeoutSeconds: 120);
}

### Inspect the raw retrieval details

Use the next cell when you want to understand how Search produced the answer or troubleshoot missing evidence. It inspects the most recent query's HTTP status (`transport.LastStatus`) and parsed response (`transport.LastResult`) in this kernel; it does not send another request or incur additional retrieval/model charges. Run a query first. If you run a different query, this cell will inspect that query instead.

- **HTTP status:** the overall request status. HTTP 200 alone does not prove that every source succeeded or that the answer is supported.
- **Full activity:** planning, source retrieval and answer-synthesis steps, including any returned counts, timings, model usage and source errors. Check which knowledge sources were actually used.
- **Web IQ activity only:** repeats just the `mcpServer` entries from the full activity so tool names, search arguments and Web IQ errors are easier to inspect. An empty list means no MCP activity was returned for this request, not necessarily an authentication failure.
- **Raw references:** the evidence objects behind `[ref_id:N]`, including returned source metadata, URLs and page fields. Compare these with the readable references above. `sourceData: null` means no source data was returned for that reference; check `includeReferenceSourceData` before assuming the document has no URL.

This is diagnostic output, not another answer. It can contain query text, internal document passages and source URLs; inspect it locally and clear sensitive outputs before sharing the notebook.

In [ ]:
KnowledgeBaseResults.Diagnostics(transport.LastResult, transport.LastStatus, settings.WebIqKey);

### 6c. A question for the indexed SharePoint documents

Change the topic to match your SharePoint documents. This example uses sustainability and AI; it does not assume the SharePoint library contains the Blob travel policy. Verify that SharePoint activity returns chunks and references, not just HTTP 200.

In [ ]:
{
    string question = $"Using {settings.SharePointSourceName}, what are the main recommendations about sustainability and AI?";
    KnowledgeBaseResults.ValidateQuery(question, outputMode);
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 60,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName,
                ["kind"] = "mcpServer",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            })
    };
    // Python uses the stored answerSynthesis default. Override only when requested.
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    await transport.SendAsync(settings, HttpMethod.Post, $"knowledgebases/{settings.BaseName}/retrieve",
        payload, userContext: true, timeoutSeconds: 120);
}

### 6d. A question that needs internal content and the web

This is the point of one knowledge base: the planner should query internal documents **and** the web, and the answer should separate the two.

In [ ]:
{
    string question = $"Using {settings.BlobSourceName} and {settings.WebIqSourceName}, how does our travel reimbursement policy compare with common corporate travel expense best practices on the public web?";
    KnowledgeBaseResults.ValidateQuery(question, outputMode);
    var payload = new JsonObject
    {
        ["messages"] = new JsonArray(new JsonObject
        {
            ["role"] = "user",
            ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
        }),
        ["includeActivity"] = true,
        ["maxRuntimeInSeconds"] = 90,
        ["knowledgeSourceParams"] = new JsonArray(
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.BlobSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.SharePointSourceName,
                ["kind"] = "searchIndex",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            },
            new JsonObject
            {
                ["knowledgeSourceName"] = settings.WebIqSourceName,
                ["kind"] = "mcpServer",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            })
    };
    // Python uses the stored answerSynthesis default. Override only when requested.
    if (outputMode != "answerSynthesis") payload["outputMode"] = outputMode;
    await transport.SendAsync(settings, HttpMethod.Post, $"knowledgebases/{settings.BaseName}/retrieve",
        payload, userContext: true, timeoutSeconds: 150);
}

## 7. Evaluate, do not just read the answer

- A fluent answer is not proof. Open `references` and confirm each claim traces to a Blob document, SharePoint item or web result. Source data contains the existing index's metadata, including SharePoint URLs and page ranges.
- Ask a question matching your SharePoint documents and confirm that `{prefix}-kb-sharepoint` appears in the activity with results. A Blob-only success does not prove SharePoint retrieval.
- Ask the same SharePoint question as two users with different document access. The existing index enforces its Entra ACLs; this notebook does not change or expand document permissions.
- Blob content has no per-user document ACLs in these workshop indexes: Search data access governs who can query it.
- Compare activity for 6a-6d. Source selection is model-driven. Adjust source descriptions and `retrievalInstructions` in section 5 if routing does not match your content.

## 8. Optional cleanup

Resources are retained by default for notebooks 12 and 14. The opt-in cell deletes only this notebook's knowledge base and its three knowledge-source definitions. It **never deletes the existing indexes or their ingestion resources**, Blob files or SharePoint documents.

A knowledge source cannot be deleted while another knowledge base references it. Remove those dependent knowledge bases first if applicable. Clear notebook outputs before sharing; they can contain internal document text and web results.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
{
    foreach (var (collection, name) in new[]
    {
        ("knowledgebases", settings.BaseName), ("knowledgesources", settings.WebIqSourceName),
        ("knowledgesources", settings.SharePointSourceName), ("knowledgesources", settings.BlobSourceName)
    })
    {
        await transport.SendAsync(settings, HttpMethod.Delete, $"{collection}/{name}", allowMissing: true);
        Console.WriteLine("Deleted or already absent: " + name);
    }
}
else Console.WriteLine("Cleanup disabled; existing indexes and knowledge resources retained.");


## Release local resources

Run this after the query/diagnostic cells. This disposes only local HTTP resources; it does not delete Azure resources. Rerun Configuration to start another local session.

In [ ]:
client.Dispose();

## References

- [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq)
- [Existing Search index knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-search-index)
- [Permission filtering at query time](https://learn.microsoft.com/azure/search/search-query-access-control-rbac-enforcement)
- [MCP server knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve from a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

Preview APIs and model availability can change.